In [0]:
import sys
import importlib

# Copy the Workspace path of your src folder and paste it here
src_path = "/Users/roy.sourav2700@gmail.com/etl_severn_trent/src"

if src_path not in sys.path:
    sys.path.insert(0, src_path)

# Clears any wrongly cached/empty common_utils package
sys.modules.pop("com_utils", None)


In [0]:
import src.com_utils
importlib.reload(src.com_utils)

print(src.com_utils.__file__)
print(dir(src.com_utils))

In [0]:
import re
from datetime import datetime

from src.com_utils import apply_scd2, run_dq_checks

catalog = "severn_trent"
silver_schema = "silver"
raw_data_path = "/Volumes/severn_trent/bronze/raw_data"

snapshot_folders = [
    item.path.rstrip("/")
    for item in dbutils.fs.ls(raw_data_path)
    if item.isDir()
    and re.fullmatch(r"\d{2}_\d{2}_\d{4}", item.name.rstrip("/"))
]

if not snapshot_folders:
    raise ValueError(f"No snapshot folders found in {raw_data_path}")

latest_snapshot_path = max(
    snapshot_folders,
    key=lambda path: datetime.strptime(path.split("/")[-1], "%d_%m_%Y")
)

print(f"Processing dimension folder: {latest_snapshot_path}")

dimensions = {
    "DimCancellationReason": {
        "join_keys": ["cancellation_id"],
        "hash_columns": [
            "cancellation_reason",
            "reason_group",
            "is_customer_driven",
            "severity"
        ]
    },
    "DimCounty": {
        "join_keys": ["county_id"],
        "hash_columns": [
            "county_name",
            "region",
            "country",
            "population_band",
            "urban_rural"
        ]
    },
    "DimPeople": {
        "join_keys": ["people_id"],
        "hash_columns": [
            "first_name",
            "surname",
            "employee_name",
            "job_role",
            "employment_type",
            "team",
            "hire_date"
        ]
    },
    "DimShrinkageType": {
        "join_keys": ["shrinkage_type_id"],
        "hash_columns": [
            "shrinkage_type",
            "category",
            "is_paid",
            "is_planned"
        ]
    }
}

for dimension_name, config in dimensions.items():
    source_path = f"{latest_snapshot_path}/{dimension_name}.csv"
    target_table = f"{catalog}.{silver_schema}.{dimension_name}"

    print(f"Processing {source_path}")
    print(f"Writing to {target_table}")

    dimension_df = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .csv(source_path)
        .dropDuplicates()
    )

    valid_df, dq_status = run_dq_checks(
      spark=spark,
      source_df=dimension_df,
      source_name=dimension_name,
      primary_keys=config["join_keys"],
      required_columns=config["join_keys"] + config["hash_columns"]
    )

    if valid_df.limit(1).count() == 0:
       print(f"No valid rows available for {dimension_name}; Silver load skipped.")
       continue

    apply_scd2(
        spark=spark,
        source_df=dimension_df,
        target_table=target_table,
        join_keys=config["join_keys"],
        hash_columns=config["hash_columns"],
        handle_deletes=False
    )

print("Dimensions processed successfully.")